# 문제 2~3: AWS S3 텍스트 WordCount

실제 AWS 키를 사용하지 않고 `moto`로 mock S3를 구성한 뒤, Spark DataFrame API로 단어별 빈도수를 계산하고 결과 CSV를 mock S3에 저장합니다.

In [ ]:
# Colab 실행 환경 준비
%pip install -q pyspark==3.5.3 boto3 moto==5.0.18

In [ ]:
import logging
import os
import sys
import tempfile
from pathlib import Path

import boto3
from moto import mock_aws
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, explode, lower, regexp_replace, split


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("problem2_3_wordcount")

# 제출 안내에 맞춰 실제 AWS 키 대신 마스킹 값을 사용합니다.
os.environ["AWS_ACCESS_KEY_ID"] = "**"
os.environ["AWS_SECRET_ACCESS_KEY"] = "**"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"


def find_example_address_file():
    candidates = [
        Path("example_address.txt"),
        Path("/content/example_address.txt"),
        Path("/Users/hanjeonghyun/Downloads/data/example_address.txt"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files

        print("example_address.txt 파일을 업로드하세요.")
        uploaded = files.upload()
        if "example_address.txt" not in uploaded:
            raise FileNotFoundError("업로드 파일명은 example_address.txt 여야 합니다.")
        return Path("example_address.txt")
    except ModuleNotFoundError as exc:
        raise FileNotFoundError("example_address.txt 파일을 현재 폴더에 두고 다시 실행하세요.") from exc


try:
    source_file = find_example_address_file()
    source_text = source_file.read_text(encoding="utf-8")
    print(f"입력 텍스트 파일: {source_file}")

    spark = (
        SparkSession.builder
        .appName("DE_5_week_problem2_3_wordcount")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark Session 생성 성공")

    with mock_aws():
        s3 = boto3.client("s3", region_name="us-east-1")
        bucket_name = "four-week-quiz"
        input_key = "example_address.txt"
        output_key = "wordcount_output_1/word_count_top10.csv"

        s3.create_bucket(Bucket=bucket_name)
        s3.put_object(
            Bucket=bucket_name,
            Key=input_key,
            Body=source_text.encode("utf-8"),
            ContentType="text/plain",
        )

        uploaded = s3.head_object(Bucket=bucket_name, Key=input_key)
        print(
            f"S3 업로드 확인: s3a://{bucket_name}/{input_key} "
            f"ContentLength={uploaded['ContentLength']}"
        )

        body = s3.get_object(Bucket=bucket_name, Key=input_key)["Body"].read().decode("utf-8")
        logger.info("S3에서 텍스트 데이터를 성공적으로 읽었습니다: s3a://%s/%s", bucket_name, input_key)

        with tempfile.TemporaryDirectory() as tmp_dir:
            work_dir = Path(tmp_dir)
            input_path = work_dir / "example_address.txt"
            output_dir = work_dir / "wordcount_output"
            input_path.write_text(body, encoding="utf-8")

            lines_df = spark.read.text(str(input_path))
            words_df = lines_df.select(
                explode(split(lower(regexp_replace(col("value"), r"[^0-9a-zA-Z가-힣\s]", " ")), r"\s+")).alias("word")
            ).where(col("word") != "")

            word_count_df = words_df.groupBy("word").count().orderBy(desc("count"), col("word"))
            top10 = word_count_df.limit(10)
            logger.info("Word Count 계산 완료")

            print("WordCount Top 10")
            top10.show(10, truncate=False)

            top10.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(output_dir))
            csv_file = next(output_dir.glob("part-*.csv"))

            s3.put_object(
                Bucket=bucket_name,
                Key=output_key,
                Body=csv_file.read_bytes(),
                ContentType="text/csv",
            )
            saved = s3.head_object(Bucket=bucket_name, Key=output_key)
            print(
                f"결과 CSV S3 저장 확인: s3a://{bucket_name}/{output_key} "
                f"ContentLength={saved['ContentLength']}"
            )
            logger.info("Word Count 결과를 S3에 저장했습니다: s3a://%s/wordcount_output_1", bucket_name)

    logger.info("WordCount 처리 성공")

except Exception as exc:
    logger.exception("WordCount 처리 실패: %s", exc)
    raise